# Customer Churn Framing, Responsible Data Audit & Non-ML Baseline Comparison

This notebook accompanies the **ML Problem-Framing Memo**, **Responsible Data Card**, and **Risk Register**.
It validates:
1. Data ingestion, completeness, and distributional sanity checks.
2. The Deterministic Non-ML Heuristic Baseline.
3. Machine Learning (Logistic Regression) comparison and probability calibration.
4. Business cost-sensitive matrix evaluation (False Negative penalty vs False Positive penalty).
5. Abstention boundary mapping and human-review queue routing.


In [1]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('customer_churn_data.csv')
print('=== Dataset Dimensions & Columns ===')
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
df


=== Dataset Dimensions & Columns ===
Rows: 12, Columns: 7


,customer_id,tenure_months,support_tickets,monthly_spend_inr,last_login_days,plan_type,churned
0,C001,2,4,499,21,Basic,1
1,C002,18,1,1299,2,Pro,0
2,C003,6,3,799,14,Basic,1
3,C004,30,0,1499,1,Pro,0
4,C005,11,2,999,5,Standard,0
5,C006,1,5,499,30,Basic,1
6,C007,24,1,999,3,Standard,0
7,C008,8,4,799,17,Basic,1
8,C009,15,2,1299,4,Pro,0
9,C010,4,3,499,12,Basic,1


## 1. Data Integrity & Summary Statistics
Verifying missing values, unique customer keys, and feature distribution across churn classes.


In [2]:
print('Missing values per column:\n', df.isnull().sum())
print('\nTarget Distribution (churned):\n', df['churned'].value_counts(normalize=True))
print('\nClass Breakdown by Plan Type:')
print(pd.crosstab(df['plan_type'], df['churned'], margins=True))
print('\nMean Feature Values Grouped by Churn Status:')
print(df.groupby('churned')[['tenure_months', 'support_tickets', 'monthly_spend_inr', 'last_login_days']].mean())


Missing values per column:
 customer_id          0
tenure_months        0
support_tickets      0
monthly_spend_inr    0
last_login_days      0
plan_type            0
churned              0
dtype: int64

Target Distribution (churned):
 churned
0    0.583333
1    0.416667
Name: proportion, dtype: float64

Class Breakdown by Plan Type:
churned    0  1  All
plan_type           
Basic      0  5    5
Pro        3  0    3
Standard   4  0    4
All        7  5   12

Mean Feature Values Grouped by Churn Status:
         tenure_months  support_tickets  monthly_spend_inr  last_login_days
churned                                                                    
0            17.857143         1.142857        1127.571429         3.714286
1             4.200000         3.800000         619.000000        18.800000


## 2. Deterministic Non-ML Heuristic Baseline
Before adopting machine learning, we formulate a simple domain-driven rule:
- Flag as at-risk if: `support_tickets >= 3` OR `last_login_days >= 14`.


In [3]:
def non_ml_baseline(row):
    return 1 if (row['support_tickets'] >= 3 or row['last_login_days'] >= 14) else 0

df['baseline_pred'] = df.apply(non_ml_baseline, axis=1)

def evaluate_classifier(y_true, y_pred, model_name='Model'):
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    beta = 2.0
    f2 = (1 + beta**2) * (precision * recall) / ((beta**2 * precision) + recall) if ((beta**2 * precision) + recall) > 0 else 0.0
    accuracy = (tp + tn) / len(y_true)
    
    # Financial Cost Model:
    # Cost of False Negative (Lost LTV) = INR 6,000
    # Cost of False Positive (Discount / outreach cost) = INR 250
    c_fn = 6000
    c_fp = 250
    total_cost = (fn * c_fn) + (fp * c_fp)
    
    print(f'=== Evaluation: {model_name} ===')
    print(f'Confusion Matrix: TP={tp}, FP={fp}, FN={fn}, TN={tn}')
    print(f'Accuracy : {accuracy:.3f}')
    print(f'Precision: {precision:.3f}')
    print(f'Recall   : {recall:.3f}')
    print(f'F1-Score : {f1:.3f}')
    print(f'F2-Score : {f2:.3f}')
    print(f'Total Business Cost: INR {total_cost:,} (FN: INR {fn * c_fn:,}, FP: INR {fp * c_fp:,})')
    print('-' * 55)
    return {'model': model_name, 'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f2': f2, 'cost': total_cost}

res_baseline = evaluate_classifier(df['churned'], df['baseline_pred'], 'Non-ML Heuristic Baseline')


=== Evaluation: Non-ML Heuristic Baseline ===
Confusion Matrix: TP=5, FP=0, FN=0, TN=7
Accuracy : 1.000
Precision: 1.000
Recall   : 1.000
F1-Score : 1.000
F2-Score : 1.000
Total Business Cost: INR 0 (FN: INR 0, FP: INR 0)
-------------------------------------------------------


## 3. Machine Learning Benchmark (Logistic Regression)
Now we evaluate a regularized classifier to extract continuous probabilities for calibrated tri-band routing.


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

features = ['tenure_months', 'support_tickets', 'monthly_spend_inr', 'last_login_days']
X = df[features]
y = df['churned']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = LogisticRegression(class_weight='balanced', random_state=42)
model.fit(X_scaled, y)

df['ml_probability'] = model.predict_proba(X_scaled)[:, 1]
df['ml_pred_default'] = (df['ml_probability'] >= 0.50).astype(int)
df['ml_pred_tuned'] = (df['ml_probability'] >= 0.35).astype(int)

res_ml_default = evaluate_classifier(df['churned'], df['ml_pred_default'], 'Logistic Regression (Threshold 0.50)')
res_ml_tuned = evaluate_classifier(df['churned'], df['ml_pred_tuned'], 'Logistic Regression (Cost-Tuned Threshold 0.35)')

# Feature weights
importance = pd.DataFrame({'feature': features, 'coefficient': model.coef_[0]}).sort_values('coefficient', ascending=False)
print('\nModel Coefficients:')
print(importance.to_string(index=False))


=== Evaluation: Logistic Regression (Threshold 0.50) ===
Confusion Matrix: TP=5, FP=0, FN=0, TN=7
Accuracy : 1.000
Precision: 1.000
Recall   : 1.000
F1-Score : 1.000
F2-Score : 1.000
Total Business Cost: INR 0 (FN: INR 0, FP: INR 0)
-------------------------------------------------------
=== Evaluation: Logistic Regression (Cost-Tuned Threshold 0.35) ===
Confusion Matrix: TP=5, FP=1, FN=0, TN=6
Accuracy : 0.917
Precision: 0.833
Recall   : 1.000
F1-Score : 0.909
F2-Score : 0.962
Total Business Cost: INR 250 (FN: INR 0, FP: INR 250)
-------------------------------------------------------

Model Coefficients:
          feature  coefficient
  support_tickets     0.839114
  last_login_days     0.811240
    tenure_months    -0.529296
monthly_spend_inr    -0.673671


## 4. Abstention & Human Review Routing Policy
Mapping predicted risk probabilities to production actions:
- p < 0.30: Low Risk (No automated retention outreach)
- 0.30 <= p < 0.65: Abstention Zone -> Human Review Queue for Customer Success Specialists
- p >= 0.65: High Risk -> Automated targeted retention workflow


In [5]:
def assign_policy(p):
    if p < 0.30:
        return 'Automated Low Risk (No Outreach)'
    elif p < 0.65:
        return 'Abstention / Human Review Queue'
    else:
        return 'Automated High Risk (Retention Offer)'

df['decision_action'] = df['ml_probability'].apply(assign_policy)
display_cols = ['customer_id', 'plan_type', 'tenure_months', 'support_tickets', 'last_login_days', 'churned', 'ml_probability', 'decision_action']
print(df[display_cols].to_string(index=False))


customer_id plan_type  tenure_months  support_tickets  last_login_days  churned  ml_probability                       decision_action
       C001     Basic              2                4               21        1        0.952634 Automated High Risk (Retention Offer)
       C002       Pro             18                1                2        0        0.044834      Automated Low Risk (No Outreach)
       C003     Basic              6                3               14        1        0.716745 Automated High Risk (Retention Offer)
       C004       Pro             30                0                1        0        0.007846      Automated Low Risk (No Outreach)
       C005  Standard             11                2                5        0        0.235205      Automated Low Risk (No Outreach)
       C006     Basic              1                5               30        1        0.988416 Automated High Risk (Retention Offer)
       C007  Standard             24                1         

## 5. Conclusions & Deployment Justification Recommendation
1. **Performance Parity:** The non-ML baseline (`support_tickets >= 3` OR `last_login_days >= 14`) achieved perfect separation on this pilot dataset.
2. **Justification Decision:** Machine Learning should **NOT** be deployed autonomously as a black box when a simple transparent rule accomplishes zero false negatives.
3. **Hybrid Path Forward:** Deploy the rule-based baseline as the operational safeguard, while logging the continuous ML score in shadow mode to route ambiguous cases (`0.30 <= p < 0.65`, e.g. Customer `C012`) to the Human Review Queue.
